## Visión general del proceso

Antes de entrar en detalle, este es el flujo completo que sigue un LLM desde que recibe texto hasta que genera una respuesta:

```mermaid
flowchart TB
    subgraph ENTRADA["1. ENTRADA"]
        TEXT["Texto: El gato duerme"]
    end
    
    subgraph TOKENIZACION["2. TOKENIZACIÓN"]
        TOKENS["Índices: [128, 4521, 892, 45]"]
    end
    
    subgraph EMBEDDINGS["3. EMBEDDINGS"]
        VECS["Vectores: matriz 4 × 4096"]
    end
    
    subgraph TRANSFORMER["4. CAPAS TRANSFORMER"]
        CAPAS["N capas de atención + feed-forward"]
    end
    
    subgraph PROYECCION["5. PROYECCIÓN"]
        LOGITS["Logits: 50000 scores"]
    end
    
    subgraph SELECCION["6. SELECCIÓN"]
        PROBS["Probabilidades + muestreo"]
    end
    
    subgraph SALIDA["7. SALIDA"]
        OUT["Índice: 237 → Texto: en"]
    end
    
    ENTRADA --> TOKENIZACION --> EMBEDDINGS --> TRANSFORMER --> PROYECCION --> SELECCION --> SALIDA
    SALIDA -.->|"Repetir hasta fin"| TOKENIZACION
    
    style TRANSFORMER fill:#9f7aea,color:#fff
    style SALIDA fill:#4ade80,color:#fff
```

| Paso | Entrada | Salida | Por qué es necesario |
|------|---------|--------|----------------------|
| **1. Tokenización** | Texto | Índices enteros | Las redes neuronales no entienden letras, solo números. Convertimos texto a IDs. |
| **2. Embeddings** | Índices | Vectores (N × dim) | Un simple ID no captura significado. Los vectores permiten representar relaciones semánticas. |
| **3. Posición** | Vectores | Vectores + posición | Sin esto, "perro muerde hombre" y "hombre muerde perro" serían idénticos para el modelo. |
| **4. Transformer** | Matriz de vectores | Matriz de vectores | Aquí ocurre la "magia": cada palabra mira a las demás para entender el contexto completo. |
| **5. Proyección** | Vector último token | Logits (vocab_size) | Convertimos la representación interna en una puntuación para cada palabra posible del vocabulario. |
| **6. Softmax** | Logits | Probabilidades | Transformamos puntuaciones arbitrarias en probabilidades que suman 1 (interpretables). |
| **7. Selección** | Probabilidades | Índice ganador | Elegimos qué palabra generar. No siempre la más probable, para añadir variedad. |
| **8. Decodificación** | Índice | Texto | Volvemos del mundo numérico al texto legible buscando el ID en la tabla del tokenizador. |

> El proceso se repite: el token generado se añade a la secuencia y se vuelve a procesar todo para predecir el siguiente token, hasta generar un token de fin o alcanzar el límite.

Las siguientes secciones explican cada paso en detalle.

## Del texto a los tokens

Cuando escribes un prompt en un LLM, el modelo no trabaja directamente con palabras o caracteres. El primer paso es la **tokenización**: convertir el texto de entrada en una secuencia de unidades discretas llamadas *tokens*.

Un token no es exactamente una palabra. Dependiendo del vocabulario del modelo, un token puede ser:

* Una palabra completa como "casa"
* Una subpalabra como "ción" o "pre"
* Un carácter individual en casos de palabras raras
* Símbolos de puntuación o espacios

> La tokenización permite que el modelo maneje cualquier texto posible, incluyendo palabras nuevas que nunca vio durante el entrenamiento, descomponiéndolas en subunidades conocidas.

```mermaid
flowchart LR
    T["El gato duerme"] --> TOK[Tokenizador]
    TOK --> T1["[El]"]
    TOK --> T2["[gato]"]
    TOK --> T3["[duer]"]
    TOK --> T4["[me]"]
    
    style TOK fill:#9f7aea,color:#fff
```

### Vocabulario y codificación

El modelo tiene un **vocabulario fijo** de tokens posibles, típicamente entre 30,000 y 100,000 tokens diferentes. Cada token tiene un identificador numérico único. El proceso de tokenización convierte el texto en una secuencia de estos identificadores.

Por ejemplo, la frase "El gato duerme" podría convertirse en la secuencia `[128, 4521, 892, 45]`, donde cada número representa un token del vocabulario. Esta secuencia numérica es lo que realmente procesa el modelo.

Los tokenizadores modernos utilizan algoritmos como **BPE** (Byte Pair Encoding) o **SentencePiece**, que aprenden a dividir el texto de forma óptima durante el entrenamiento. Estos algoritmos equilibran entre tener tokens muy frecuentes (palabras comunes enteras) y tokens que permitan representar cualquier texto posible.

## De tokens a vectores

Una vez tokenizado el texto, cada identificador de token se convierte en un **vector de embeddings**: una representación numérica en un espacio de alta dimensión. Este espacio puede tener cientos o miles de dimensiones.

```mermaid
flowchart TB
    subgraph Entrada["Secuencia de tokens"]
        ID1["128"] 
        ID2["4521"]
        ID3["892"]
    end
    
    subgraph Embeddings["Matriz de embeddings"]
        E[Tabla de búsqueda<br/>vocab_size x dim]
    end
    
    subgraph Vectores["Vectores resultantes"]
        V1["[0.2, -0.5, 0.8, ...]"]
        V2["[0.1, 0.3, -0.2, ...]"]
        V3["[-0.4, 0.7, 0.1, ...]"]
    end
    
    ID1 --> E --> V1
    ID2 --> E --> V2
    ID3 --> E --> V3
    
    style E fill:#9f7aea,color:#fff
```

### El significado en los embeddings

Los embeddings no son asignaciones aleatorias. Durante el entrenamiento, el modelo aprende a ubicar tokens con **significados similares** cerca unos de otros en este espacio vectorial.

* "Rey" y "reina" estarán próximos porque comparten contextos de uso.
* "Perro" y "gato" estarán más cerca entre sí que "perro" y "democracia".
* Las relaciones semánticas se codifican como direcciones: "rey - hombre + mujer ≈ reina".

> Los embeddings capturan no solo similitud semántica, sino también relaciones gramaticales, asociaciones contextuales y patrones de uso del lenguaje.

### Codificación posicional

Un problema de los Transformers es que procesan todos los tokens en paralelo, perdiendo la noción de orden. Para solucionar esto, se añade **información posicional** a cada embedding.

Esta codificación posicional indica al modelo dónde está cada token en la secuencia. Sin ella, el modelo no distinguiría entre "el perro mordió al hombre" y "el hombre mordió al perro".

Los modelos modernos utilizan diferentes estrategias:

* **Posicional absoluta**: cada posición tiene un vector único predefinido.
* **Posicional relativa** (RoPE, ALiBi): codifica la distancia entre tokens, permitiendo generalizar a secuencias más largas que las vistas durante el entrenamiento.

## El procesamiento por capas

Con los tokens convertidos en vectores posicionados, comienza el procesamiento real. Un LLM consta de múltiples **capas de Transformer** apiladas, donde cada capa refina la representación de los tokens. Los modelos actuales pueden tener desde decenas hasta más de cien capas.

```mermaid
flowchart TB
    E[Embeddings + posición] --> L1[Capa 1]
    L1 --> L2[Capa 2]
    L2 --> L3[...]
    L3 --> LN[Capa N]
    LN --> O[Representación final]
    
    subgraph Capa["Estructura de cada capa"]
        N1[Layer Norm] --> A[Self-Attention]
        A --> R1((+))
        N2[Layer Norm] --> F[Feed-Forward]
        F --> R2((+))
    end
    
    style Capa fill:#e8f4f8
    style O fill:#9f7aea,color:#fff
    style R1 fill:#4ade80,color:#fff
    style R2 fill:#4ade80,color:#fff
```

Cada capa contiene dos bloques principales: **self-attention** y **feed-forward**. Pero hay dos componentes adicionales que son fundamentales para que el modelo funcione: las **conexiones residuales** y la **normalización de capas**.

### Conexiones residuales

Las **conexiones residuales** (representadas por los símbolos + en el diagrama) suman la entrada de cada bloque a su salida. En lugar de transformar completamente la representación, cada bloque solo necesita aprender el *cambio* necesario.

> Sin conexiones residuales, los gradientes se desvanecen en redes profundas y el entrenamiento falla. Estas conexiones permiten que la información fluya directamente a través de muchas capas.

Este patrón tiene dos beneficios clave:

* Permite entrenar modelos con decenas o cientos de capas sin que los gradientes desaparezcan.
* Cada capa puede optar por *no hacer nada* si la representación actual ya es adecuada.

### Normalización de capas

La **Layer Normalization** estabiliza el entrenamiento normalizando las activaciones en cada posición. Los modelos modernos aplican la normalización *antes* de cada bloque (configuración conocida como Pre-LN), lo que mejora la estabilidad durante el entrenamiento.

### Self-Attention con máscara causal

En cada capa, el mecanismo de **self-attention** permite que cada token *atienda* a todos los tokens anteriores en la secuencia. Este proceso calcula qué tokens son relevantes para entender cada posición.

Para evitar que el modelo "haga trampa" mirando tokens futuros durante el entrenamiento, se aplica una **máscara causal**: una matriz triangular que bloquea la atención hacia posiciones posteriores.

```mermaid
flowchart LR
    subgraph Mascara["Máscara causal"]
        direction TB
        M["Token 1: ve [1]<br/>Token 2: ve [1,2]<br/>Token 3: ve [1,2,3]<br/>Token 4: ve [1,2,3,4]"]
    end
    
    style Mascara fill:#e8f4f8
```

Para una frase como "El banco cerca del río estaba vacío", cuando el modelo procesa "vacío":

* Atiende a "banco" y a "río" para entender que se refiere a un asiento junto al agua, no a una entidad financiera.
* La atención asigna pesos altos a las palabras relevantes y bajos a las irrelevantes.
* Este proceso se repite para cada token, creando representaciones contextualizadas.

> La atención es lo que permite al modelo entender que la misma palabra tiene significados diferentes según el contexto, resolviendo ambigüedades de forma dinámica.

### Múltiples cabezas de atención

Los modelos utilizan **multi-head attention**: múltiples mecanismos de atención operando en paralelo. Cada "cabeza" puede especializarse en diferentes tipos de relaciones:

* Una cabeza puede capturar relaciones sintácticas (sujeto-verbo).
* Otra puede enfocarse en relaciones semánticas (sinónimos, antónimos).
* Otra puede detectar referencias a entidades mencionadas previamente.

Las salidas de todas las cabezas se concatenan y se proyectan de nuevo a la dimensión original, permitiendo al modelo capturar múltiples tipos de dependencias simultáneamente.

### Red Feed-Forward

Después de la atención, cada token pasa por una **red feed-forward** independiente. Esta red tiene una estructura característica: primero expande la dimensión del vector (típicamente 4 veces), aplica una función de activación no lineal, y luego reduce de nuevo a la dimensión original.

```mermaid
flowchart LR
    IN["Vector<br/>dim=d"] --> E["Expansión<br/>dim=4d"]
    E --> ACT["Activación<br/>GELU/SiLU"]
    ACT --> C["Contracción<br/>dim=d"]
    C --> OUT["Vector<br/>dim=d"]
    
    style ACT fill:#9f7aea,color:#fff
```

Las funciones de activación modernas como **GELU** o **SiLU** han reemplazado a ReLU en los LLM, proporcionando transiciones más suaves que mejoran el aprendizaje.

La combinación de atención (que mezcla información entre posiciones) y feed-forward (que procesa cada posición) da a los Transformers su capacidad de modelar tanto relaciones contextuales como transformaciones locales.

## El flujo de datos entre capas

Antes de ver la predicción final, es importante entender qué tipo de datos fluyen por el modelo. Cada capa transformer recibe y produce **tensores de la misma forma**: matrices de dimensión `(longitud_secuencia × dimensión_modelo)`.

```mermaid
flowchart LR
    subgraph Entrada["Entrada a capa N"]
        E1["Token 1: vector dim=4096"]
        E2["Token 2: vector dim=4096"]
        E3["Token 3: vector dim=4096"]
    end
    
    CAPA[Capa Transformer N]
    
    subgraph Salida["Salida de capa N"]
        S1["Token 1: vector dim=4096"]
        S2["Token 2: vector dim=4096"]
        S3["Token 3: vector dim=4096"]
    end
    
    Entrada --> CAPA --> Salida
    
    style CAPA fill:#9f7aea,color:#fff
```

Si el modelo tiene dimensión 4096, cada token está representado por un vector de 4096 números reales. Tres tokens forman una matriz de 3×4096. Esta estructura se mantiene idéntica a través de todas las capas.

> Lo que cambia entre capas no es la forma de los datos, sino su contenido. Cada capa enriquece los vectores con más información contextual, refinando la representación.

Al salir de la última capa, cada vector contiene información "destilada" de toda la secuencia. El vector del último token es especialmente importante: acumula todo el contexto necesario para predecir qué viene después.

## Predicción del siguiente token

Tras pasar por todas las capas, el modelo tiene una representación final para cada posición. Para generar texto, utiliza la representación del **último token** para predecir el siguiente.

```mermaid
flowchart LR
    R["Vector final<br/>dim=4096"] --> P["Matriz de proyección<br/>4096 × 50000"]
    P --> L["Logits<br/>50000 valores"]
    L --> S[Softmax]
    S --> PROB["Probabilidades<br/>50000 valores"]
    PROB --> SEL["Índice seleccionado<br/>ej: 892"]
    SEL --> TOK["Tokenizador"]
    TOK --> OUT["Texto: duer"]
    
    style L fill:#9f7aea,color:#fff
    style SEL fill:#4ade80,color:#fff
```

### De representación a probabilidades

El vector de representación final (por ejemplo, 4096 dimensiones) se multiplica por una **matriz de proyección** de tamaño `(dimensión × vocab_size)`. El resultado son los **logits**: un vector con tantos valores como tokens hay en el vocabulario.

Los logits no son embeddings ni probabilidades. Son *scores* sin normalizar, donde valores más altos indican tokens más probables. Estos logits se convierten en **probabilidades** mediante la función softmax, que los normaliza para que sumen 1.

Si el vocabulario tiene 50,000 tokens, el modelo produce 50,000 probabilidades, una para cada token posible como continuación. Por ejemplo:

| Token | Probabilidad |
|-------|--------------|
| "feliz" | 0.15 |
| "contento" | 0.12 |
| "triste" | 0.08 |
| ... | ... |

### Estrategias de selección

El modelo no siempre elige el token más probable. Diferentes **estrategias de muestreo** controlan cómo se selecciona el siguiente token:

* **Greedy**: selecciona siempre el token más probable. Produce texto determinista pero puede ser repetitivo.

* **Sampling con temperatura**: añade aleatoriedad controlada. Temperatura alta produce texto más creativo pero menos coherente; temperatura baja, más predecible.

* **Top-k**: limita la selección a los k tokens más probables, descartando opciones improbables.

* **Top-p (nucleus)**: selecciona entre los tokens cuyas probabilidades acumuladas sumen p, adaptándose dinámicamente a la distribución.

> La temperatura y los parámetros de muestreo permiten controlar el equilibrio entre creatividad y coherencia en las respuestas del modelo.

### Del índice al texto

Una vez seleccionado un índice (por ejemplo, 892), el tokenizador lo convierte a texto usando su **tabla de vocabulario**. Esta tabla es simplemente un mapeo bidireccional:

| Índice | Texto |
|--------|-------|
| 128 | "El" |
| 4521 | "gato" |
| 892 | "duer" |
| 45 | "me" |

No hay ninguna "desconversión" de embeddings. El modelo nunca trabaja con texto internamente, solo con índices y vectores. La conversión final es una simple búsqueda en la tabla del tokenizador.

> El embedding de un token se usa para procesarlo dentro del modelo. El índice del token se usa para convertirlo a texto. Son dos representaciones diferentes del mismo token.

### Reconstrucción de palabras

Como los tokens no siempre son palabras completas, el texto final se forma **concatenando** los fragmentos generados. El tokenizador sabe cómo unirlos correctamente:

```mermaid
flowchart LR
    I1["Índice 892"] --> T1["duer"]
    I2["Índice 45"] --> T2["me"]
    T1 --> C["Concatenar"]
    T2 --> C
    C --> FINAL["duerme"]
    
    style FINAL fill:#4ade80,color:#fff
```

Algunos tokenizadores usan marcadores especiales para indicar si un token es inicio de palabra o continuación. Por ejemplo, en algunos sistemas "duer" podría almacenarse como "duer" y "me" como "##me", indicando que es continuación. Otros simplemente concatenan y el resultado es texto válido.

### Generación autoregresiva

El proceso de generación es **iterativo**. Una vez seleccionado un token, se añade a la secuencia y el modelo procesa todo de nuevo para predecir el siguiente token.

```mermaid
flowchart LR
    P1["Prompt: El gato"] --> M1[Modelo]
    M1 --> T1["duerme"]
    
    P2["El gato duerme"] --> M2[Modelo]
    M2 --> T2["en"]
    
    P3["El gato duerme en"] --> M3[Modelo]
    M3 --> T3["el"]
    
    style M1 fill:#9f7aea,color:#fff
    style M2 fill:#9f7aea,color:#fff
    style M3 fill:#9f7aea,color:#fff
```

Este proceso continúa hasta que:

* El modelo genera un token especial de fin de secuencia.
* Se alcanza un límite máximo de tokens configurado.
* El usuario interrumpe la generación.

Cada token generado influye en los siguientes, ya que forma parte del contexto que el modelo utiliza para sus predicciones. Esto explica por qué pequeños cambios en el prompt pueden producir respuestas muy diferentes: alteran las probabilidades de los primeros tokens, que a su vez afectan a toda la cadena posterior.

### KV Cache: eficiencia en la generación

Si el modelo tuviera que reprocesar toda la secuencia desde cero para cada nuevo token, la generación sería extremadamente lenta. Para evitar esto, los LLM utilizan una técnica llamada **KV Cache** (Key-Value Cache).

Durante el cálculo de atención, cada token produce vectores de *keys* y *values*. Una vez calculados, estos vectores no cambian cuando se añaden nuevos tokens al final de la secuencia. El KV Cache almacena estos vectores para reutilizarlos.

```mermaid
flowchart TB
    subgraph Cache["KV Cache"]
        K1["K1, V1"]
        K2["K2, V2"]
        K3["K3, V3"]
    end
    
    NEW["Token nuevo"] --> Q["Query"]
    Q --> ATT["Atención"]
    Cache --> ATT
    ATT --> OUT["Salida"]
    
    style Cache fill:#e8f4f8
    style NEW fill:#9f7aea,color:#fff
```

Con KV Cache, generar el token N+1 solo requiere:

* Calcular el embedding del nuevo token.
* Computar su query y atender a las keys/values almacenadas.
* Añadir sus propias keys/values al cache para futuros tokens.

> El KV Cache reduce la complejidad de O(n) a O(1) por cada nuevo token generado, haciendo viable la generación de respuestas largas en tiempo razonable.

Esta optimización es la razón por la que los modelos distinguen entre la fase de *prefill* (procesar el prompt inicial, que es lenta) y la fase de *decode* (generar tokens uno a uno, que es rápida gracias al cache).